## Feature Selection for Asteroid Diameter Prediction

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load data
filepath = '/content/drive/MyDrive/Data/dataset.csv'
df = pd.read_csv(filepath)

print(f"Original dataset shape: {df.shape}")
print(f"\nColumns in dataset: {len(df.columns)}")

In [ ]:
# Step 1: Define features to EXCLUDE (non-predictive columns)
print("="*70)
print("STEP 1: IDENTIFY AND REMOVE NON-PREDICTIVE COLUMNS")
print("="*70)

exclude_cols = [
    # Identifiers - not useful for prediction
    'id', 'spkid', 'full_name', 'pdes', 'name', 'prefix', 'orbit_id',
    
    # Target variable - cannot be used as input
    'diameter',
    
    # Measurement uncertainty - correlates too much with diameter availability
    'diameter_sigma',
    
    # Redundant time variables (keep only epoch_mjd)
    'epoch', 'epoch_cal', 'tp_cal',
    
    # Derived orbital elements (keep base orbital elements)
    'per', 'per_y',  # Orbital period (derived from semi-major axis)
    
    # Measurement noise/sigma columns - add noise without direct predictive power
    'sigma_e', 'sigma_a', 'sigma_q', 'sigma_i', 'sigma_om',
    'sigma_w', 'sigma_ma', 'sigma_ad', 'sigma_n', 'sigma_tp', 'sigma_per'
]

print(f"\nColumns to EXCLUDE ({len(exclude_cols)}):")
for i, col in enumerate(exclude_cols, 1):
    print(f"  {i:2d}. {col:20s} - {['Identifier', 'Target', 'Uncertainty', 'Time', 'Derived', 'Noise'][min(i-1, 5)]}")

df_clean = df.drop(columns=exclude_cols, errors='ignore')
print(f"\nRemaining columns after exclusion: {len(df_clean.columns)}")
print(f"New dataset shape: {df_clean.shape}")

In [ ]:
# Step 2: Categorize remaining features
print("\n" + "="*70)
print("STEP 2: CATEGORIZE REMAINING FEATURES")
print("="*70)

print(f"\nRemaining columns ({len(df_clean.columns)}):")
print(df_clean.columns.tolist())

# Categorize by type
brightness_features = ['H', 'albedo']  # Related to size
orbital_features = ['a', 'e', 'q', 'i', 'om', 'w', 'ma', 'ad', 'n', 'tp']  # Orbital elements
risk_features = ['neo', 'pha', 'moid', 'moid_ld']  # NEO/hazard related
class_features = [col for col in df_clean.columns if col.startswith('class_')]  # Asteroid class
other_features = ['equinox', 'rms']  # Other

print(f"\nBrightness features ({len(brightness_features)}): {brightness_features}")
print(f"Orbital features ({len(orbital_features)}): {orbital_features}")
print(f"Risk/NEO features ({len(risk_features)}): {risk_features}")
print(f"Class features ({len(class_features)}): Asteroid taxonomic classes")
print(f"Other features ({len(other_features)}): {other_features}")

In [ ]:
# Step 3: Encode categorical variables
print("\n" + "="*70)
print("STEP 3: ENCODE CATEGORICAL VARIABLES")
print("="*70)

df_encoded = df_clean.copy()

# Encode neo and pha (Y/N to 1/0)
print("\nEncoding neo and pha columns:")
df_encoded['neo'] = df_encoded['neo'].map({'Y': 1, 'N': 0})
df_encoded['pha'] = df_encoded['pha'].map({'Y': 1, 'N': 0})
print(f"  neo: Y/N → 1/0")
print(f"  pha: Y/N → 1/0")

# Encode equinox
if 'equinox' in df_encoded.columns:
    print(f"\nEncoding equinox:")
    df_encoded = pd.get_dummies(df_encoded, columns=['equinox'], drop_first=True)
    print(f"  One-hot encoded (drop_first=True to avoid multicollinearity)")

print(f"\nDataset shape after encoding: {df_encoded.shape}")
print(f"Data types:\n{df_encoded.dtypes}")

In [ ]:
# Step 4: Handle missing values
print("\n" + "="*70)
print("STEP 4: ASSESS MISSING VALUES")
print("="*70)

missing_pct = (df_encoded.isnull().sum() / len(df_encoded) * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

print(f"\nColumns with missing values:")
for col, pct in missing_pct.items():
    print(f"  {col:25s}: {pct:6.2f}%")

print(f"\nStrategy:")
print(f"  - Drop rows with missing values to ensure clean training data")
print(f"  - Focus on complete cases")

df_final = df_encoded.dropna()
print(f"\nRecords after removing missing values: {len(df_final):,} (was {len(df_encoded):,})")
print(f"Data retention: {len(df_final)/len(df_encoded)*100:.2f}%")

In [ ]:
# Step 5: Final feature selection
print("\n" + "="*70)
print("STEP 5: SELECT MEANINGFUL INPUT FEATURES")
print("="*70)

# Separate features and target
X = df_final.drop(columns=['diameter'])  # diameter is the target
y = df_final['diameter']

print(f"\nTarget variable (y): diameter")
print(f"  - Shape: {y.shape}")
print(f"  - Mean: {y.mean():.2f} km")
print(f"  - Range: {y.min():.3f} - {y.max():.2f} km")

print(f"\nInput features (X): {len(X.columns)} columns")
print(f"  - Shape: {X.shape}")
print(f"\nSelected features:")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:2d}. {col}")

In [ ]:
# Step 6: Feature importance analysis
print("\n" + "="*70)
print("STEP 6: ASSESS FEATURE IMPORTANCE")
print("="*70)

# Train a quick Random Forest to assess feature importance
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1, max_depth=20)
rf_model.fit(X, y)

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nFeature Importance Ranking (Random Forest):")
for idx, row in feature_importance.iterrows():
    bar = '█' * int(row['importance'] * 100)
    print(f"  {row['feature']:25s}: {row['importance']:7.4f} {bar}")

top_features = feature_importance.head(10)['feature'].tolist()
print(f"\nTop 10 most important features:")
for i, feat in enumerate(top_features, 1):
    imp = feature_importance[feature_importance['feature'] == feat]['importance'].values[0]
    print(f"  {i:2d}. {feat:25s} (importance: {imp:.4f})")

In [ ]:
# Visualize feature importance
fig, ax = plt.subplots(figsize=(12, 8))

top_20 = feature_importance.head(20)
colors = plt.cm.viridis(np.linspace(0, 1, len(top_20)))
ax.barh(range(len(top_20)), top_20['importance'].values, color=colors)
ax.set_yticks(range(len(top_20)))
ax.set_yticklabels(top_20['feature'].values, fontsize=10)
ax.set_xlabel('Feature Importance', fontsize=12, fontweight='bold')
ax.set_title('Top 20 Most Important Features for Diameter Prediction', fontsize=13, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

for i, v in enumerate(top_20['importance'].values):
    ax.text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\nFeature importance visualization saved.")

In [ ]:
# Step 7: Summary and recommendations
print("\n" + "="*70)
print("STEP 7: SUMMARY AND RECOMMENDATIONS")
print("="*70)

print(f"\n✓ FINAL INPUT FEATURE SET:")
print(f"\n  Total features selected: {len(X.columns)}")
print(f"  Training samples: {len(X):,}")

print(f"\n✓ FEATURE CATEGORIES:")
print(f"  1. Brightness Features: H, albedo")
print(f"  2. Orbital Elements: a, e, q, i, om, w, ma, ad, n, tp")
print(f"  3. NEO/Risk Indicators: neo, pha, moid, moid_ld")
print(f"  4. Asteroid Classification: class_APO, class_AST, ... (12 classes)")
print(f"  5. Other: equinox (encoded), rms")

print(f"\n✓ TOP 5 MOST IMPORTANT FEATURES:")
for i, row in feature_importance.head(5).iterrows():
    print(f"  {row['feature']:25s}: {row['importance']:.4f}")

print(f"\n✓ RECOMMENDATIONS FOR MODELING:")
print(f"  • Use all {len(X.columns)} features for initial model training")
print(f"  • For feature reduction: keep top 15-20 features based on importance")
print(f"  • No scaling needed for tree-based models (Random Forest, XGBoost, LightGBM)")
print(f"  • Scale features for linear models (regression, neural networks)")
print(f"  • Consider feature engineering: H + albedo interaction terms")
print(f"  • Consider log-transformation of target (diameter) for regression")

print(f"\n" + "="*70)

In [ ]:
# Export prepared dataset
print("\n" + "="*70)
print("STEP 8: EXPORT PREPARED DATA")
print("="*70)

print(f"\nExporting X (features) and y (target)...")
print(f"\nFeatures (X):")
print(X.head())
print(f"\nTarget (y):")
print(y.head())

print(f"\nData ready for model training!")
print(f"  - X.shape: {X.shape}")
print(f"  - y.shape: {y.shape}")
print(f"  - No missing values")
print(f"  - All categorical variables encoded")
print(f"  - All non-predictive columns removed")